In [1]:
import operations_pb2_grpc
from app.configuration.Settings import Settings
!pip3 install -r ../requirements.txt

  Using cached numpy-1.26.4-cp313-cp313-macosx_15_0_arm64.whl
  Using cached pandas-2.0.3.tar.gz (5.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pandas (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [2821 lines of output]
      <string>:19: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
      /private/var/folders/3h/nl65v6r16_97nthmtcqbds4c0000gn/T/pip-build-env-emqmqr6l/overlay/lib/python3.13/site-packages/setuptools/config/_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
      !!
      
              ********************************************************************************
              Please use a simple string containing a SPDX expression for `project.license`. You can al

In [7]:
invest_api_key = input("Enter Invest API Key: ")
SBER_INSTRUMENT_ID = "e6123145-9665-43e0-8413-cd61b8aa9b13"
instrument_id = SBER_INSTRUMENT_ID

In [2]:
from ml.oneMinute.LSTM.configuration.LstmConfiguration import LstmConfiguration
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lstm_configuration = LstmConfiguration(
    input_size=1,
    hidden_layer_size=32,
    num_layers=1,
    output_size=1
)


best_sell_signal = 0.0012
best_buy_signal = 0.0013

In [12]:
from ml.runner.StockTrader import StockTrader
from ml.runner.configuration.TradingConfiguration import TradingConfiguration


trading_configuration = TradingConfiguration(
    sell_signal=best_sell_signal,
    buy_signal=best_buy_signal,
)

trader = StockTrader(
    model_configuration=lstm_configuration,
    device=device,
    trading_configuration=trading_configuration,
)

In [13]:
trader.load(
    "../savedModels/LSTM_ONE_MINUTE.pth",
    "../savedModels/LSTM_ONE_MINUTE_normalizer.txt",
)

In [14]:
from ml.oneMinute.LSTM.simulation.Simulator import TradingSimulator


simulator = TradingSimulator(
    invest_api_key=invest_api_key,
    model=trader.model,
    instrument_id=instrument_id,
    scaler=trader.scaler
)

In [15]:
from datetime import datetime, timedelta, timezone


start_timestamp = datetime.now(timezone.utc) - timedelta(days=30)
end_timestamp = datetime.now(timezone.utc) - timedelta(days=0)

await simulator.simulate_trading(
        device=trader.device,
        start_timestamp_utc=start_timestamp,
        end_timestamp_utc=end_timestamp,
        initial_balance=10000,
        commission=0.0005,
        lookback=16,
        buy_signal=trader.trading_configuration.buy_signal,
        sell_signal=trader.trading_configuration.sell_signal,
)

2025-05-03 14:15:29,757 - [SIMULATOR] - INFO - === Trading Simulation Results ===
2025-05-03 14:15:29,757 - [SIMULATOR] - INFO - SimulationResult(initial_balance=10000, final_balance=np.float64(15059.650500000012), total_return=np.float64(0.5059650500000012), annualized_sharpe_ratio=np.float64(605.817655983508), total_trades=1651, commission_paid=np.float64(8190.973699999998))


SimulationResult(initial_balance=10000, final_balance=np.float64(15059.650500000012), total_return=np.float64(0.5059650500000012), annualized_sharpe_ratio=np.float64(605.817655983508), total_trades=1651, commission_paid=np.float64(8190.973699999998))

In [74]:
from grpc import aio, ssl_channel_credentials
from externalClients.TInvestApi.proto import (
    users_pb2, users_pb2_grpc,
    operations_pb2, operations_pb2_grpc,
    instruments_pb2, instruments_pb2_grpc,
)

endpoint = "invest-public-api.tinkoff.ru:443"
credentials = ssl_channel_credentials()

channel = aio.secure_channel(endpoint, credentials)
users_stub = users_pb2_grpc.UsersServiceStub(channel)
operations_stub = operations_pb2_grpc.OperationsServiceStub(channel)
instrument_stub = instruments_pb2_grpc.InstrumentsServiceStub(channel)

In [91]:
from externalClients.TInvestApi.proto import (
    common_pb2,
)

NANO_CONVERSION_FACTOR = 10e-9
def quotation_to_float(quotation: common_pb2.Quotation) -> float:
    return quotation.units + quotation.nano * NANO_CONVERSION_FACTOR

def get_metadata(api_key):
    return [('authorization', f'Bearer {api_key}')]

async def get_accounts(stub):
    request = users_pb2.GetAccountsRequest(status=4)
    response = await stub.GetAccounts(request, metadata=get_metadata(invest_api_key))
    return response

async def get_portfolio(stub, user_id: str):
    request = operations_pb2.PortfolioRequest(
        account_id=user_id,
        currency="RUB"
    )
    response = await stub.GetPortfolio(request, metadata=get_metadata(invest_api_key))
    return response

async def get_instrument(stub, uid: str):
    request = instruments_pb2.InstrumentRequest(
        id_type=instruments_pb2.InstrumentIdType.INSTRUMENT_ID_TYPE_UID,
        id=uid
    )
    response = await stub.ShareBy(request, metadata=get_metadata(invest_api_key))
    return response

In [90]:
from app.configuration.Settings import Settings


settings = Settings(config_file="../../app/appsettings.json")
supported_instruments = settings.get("SupportedInstruments")['Shares']
supported_instruments

['e6123145-9665-43e0-8413-cd61b8aa9b13',
 '962e2a95-02a9-4171-abd7-aa198dbe643a',
 '509edd0c-129c-4ee2-934d-7f6246126da1',
 '7de75794-a27f-4d81-a39b-492345813822',
 '02cfdf61-6298-4c0f-a9ca-9cabc82afaf3']

In [95]:
uid_to_figi = {}
for instrument_id in supported_instruments:
    instrument_info = await get_instrument(instrument_stub, instrument_id)
    uid_to_figi[instrument_id] = instrument_info.instrument.figi

In [ ]:
account = (await get_accounts(users_stub)).accounts[0]
account_id = account.id

In [72]:
portfolio = await get_portfolio(operations_stub, account_id)
assets = {asset.figi: quotation_to_float(asset.quantity) for asset in portfolio.positions}
rub = assets['RUB000UTSTOM']
sber = assets[uid_to_figi[SBER_INSTRUMENT_ID]]